<a href="https://colab.research.google.com/github/navneetjuneja26/30MinTask/blob/master/Great_Learning_Assignment_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**The Notebook is in "view only" mode. Make a copy of this Notebook and start solving.**

# **Assignment:**

# **Question_1: Domain Adaptation & Fine-Tuning:**


**The Scenario:** You are tasked with deploying a text classifier. You must fine-tune a pre-trained Large Language Model on a dataset of 10,000 instances.

**The Constraint:** Your computing environment is strictly limited to a single 16GB VRAM GPU (e.g., Google Colab's default Tesla T4).

Throughout this notebook, you will face architectural and configuration decisions. Read the Markdown prompts carefully, choose the appropriate choice, and uncomment the corresponding code to proceed.

# **INSTRUCTIONS:**


Read the following guidelines carefully before proceeding. **No clarification will be entertained as this is a graded assignment. Make appropriate assumptions as needed and mention it clearly**

### 1. Decision-Making & Defense
Throughout this notebook, you will encounter multiple-choice architectural and configuration decisions (labeled A, B, etc.).
* Depending on the specific constraint or task requirements, **sometimes only one choice is correct**, and **other times both choices are fully applicable** with different trade-offs.
* For every decision point, you must **write a brief markdown defense** right below the code cell explaining *why* you selected your specific choice A or B.

### 2. Exploring Alternative Paths
If you wish to experiment with **both** Path A and Path B to compare their performance or observations:
* Do not overwrite your primary working solution.
* Create a separate copy of this notebook and name the file using the following naming convention:
  `[YourRollNumber]_[Decision_Number]_[Choice_Chosen].ipynb` *(e.g., `12345_Decision_1_B.ipynb`)*

In [ ]:
# Install required libraries
!pip install -q transformers datasets peft torch evaluate scikit-learn

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import get_peft_model, LoraConfig, TaskType
!pip install --upgrade torchao
!pip install -q "datasets<5.0.0" transformers peft torch evaluate scikit-learn

# Verify GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# **Decision 1: Base Model**
To classify text effectively, you need a model that understands language deeply. However, model size directly impacts memory consumption.

**A: `roberta-base` (~125 Million Parameters)**


**B: `FacebookAI/xlm-roberta-xxl` (~10.7 Billion Parameters)**


**Your Task:** Select the model that will successfully load and train within your collab notebook. Uncomment the `model_name` for your chosen path in the code cell below.


In [ ]:
# =========================================================
# PATH A: Standard Scale Encoder
# =========================================================
# model_name = "roberta-base"

# =========================================================
# PATH B: Massive Scale Encoder
# =========================================================
# model_name = "FacebookAI/xlm-roberta-xxl"

# ---------------------------------------------------------
# DO NOT MODIFY BELOW THIS LINE
# ---------------------------------------------------------
if 'model_name' not in locals():
    raise ValueError("Please uncomment a model_name path above!")

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the base model without the pre-trained head
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
).to(device)

print(f"Successfully loaded {model_name}!")

# **Decision 1 - Justification (Fill this out)**




**Which option you choose (A or B):**


**Why you choosed this option:** [Provide your technical defense]


**Why you rejected the other option:** [Explain why the alternative failed or was suboptimal for this scenario]

# **Decision 2: Tokenization Strategy Selection**
Before passing raw text into the model, it must be decomposed into subwords and mapped to integer token IDs.

---

### A: Byte-Level BPE


### B: SentencePiece / Unigram


### C: WordPiece


### D: SentencePiece with Sentinel Formatting


**Your Task:** Select the explicit tokenizer that corresponds to your chosen base model from Decision 1. Uncomment your choice in the cell below.

In [ ]:
from transformers import (
    RobertaTokenizer,
    XLMRobertaTokenizer,
    BertTokenizer,
    T5Tokenizer
)

# =========================================================
# A: Byte-Level BPE (RoBERTa)
# =========================================================
# tokenizer = RobertaTokenizer.from_pretrained(model_name)

# =========================================================
# B: SentencePiece / Unigram (XLM-RoBERTa)
# =========================================================
# tokenizer = XLMRobertaTokenizer.from_pretrained(model_name)

# =========================================================
# C: WordPiece (BERT)
# =========================================================
# tokenizer = BertTokenizer.from_pretrained(model_name)

# =========================================================
# D: SentencePiece with Sentinels (T5)
# =========================================================
# tokenizer = T5Tokenizer.from_pretrained(model_name)

# ---------------------------------------------------------
# DO NOT MODIFY BELOW THIS LINE
# ---------------------------------------------------------
if 'tokenizer' not in locals():
    raise ValueError("Please uncomment one of the tokenizer paths above!")

# Verify and assign padding token if absent
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loaded Tokenizer Type : {tokenizer.__class__.__name__}")
print(f"Vocabulary Size       : {tokenizer.vocab_size:,}")
print(f"Pad Token / ID        : {tokenizer.pad_token} ({tokenizer.pad_token_id})")

# **Decision 2 - Justification (Fill this out)**
**Which Tokenizer you choose (A or B or C or D):**

**Why you choosed this option:** [Provide your technical defense]

**Why you rejected the other option:** [Explain why the alternative failed or was suboptimal for this scenario]

# **Decision 3: Fine-Tuning Strategy Selection**
Specialized domain classification introduces terminology and syntactic patterns not heavily emphasized in general pre-training. You must select how to update the network.

**A: Head-Only Fine-Tuning (Run cell  4)**
Freeze the entire pre-trained backbone and only update the newly initialized classification layers at the top.


**B: Low-Rank Adaptation (LoRA) (Run cell  5)**
Freeze the base model weights, but inject trainable rank-decomposition matrices into key attention projection layers.


**Instructions:** Run **EITHER** Cell 4 (A) **OR** Cell 5 (B). Do not run both. Afterward, execute Cell 6 to verify the resulting trainable parameter footprint.

In [ ]:
###Cell - 4

# =========================================================
# A: Head-Only Fine-Tuning
# Run this cell ONLY if you are selecting A
# =========================================================

# Freeze all backbone layers; keep only classifier/score parameters trainable
for name, param in base_model.named_parameters():
    if "classifier" not in name and "score" not in name:
        param.requires_grad = False

model = base_model
fine_tuning_strategy = "head_only"
print("Strategy Applied: Head-Only Fine-Tuning (Backbone frozen).")

# **Answer this  question, Only if you decides to do LORA finetuning**

# **Decision 3.1: LoRA Target Module Placement**
You must determine which internal Transformer projection matrices will receive low-rank adapter weights.

**A: Attention-Only Projections (`query`, `value`)**
Injects rank-decomposition matrices strictly into the Query and Value projections of the self-attention blocks.


**B: Comprehensive Injection (Attention + Feed-Forward Networks)**
Injects rank-decomposition matrices into all linear transformations, including self-attention (`query`, `key`, `value`) and the MLP/intermediate feed-forward blocks (`dense`).


**Instructions:** Select your target module strategy by uncommenting the appropriate block in the code cell below.

In [ ]:
## Cell 5

# =========================================================
# A: Attention-Only Projections (Minimalist)
# =========================================================
# target_mods = ["query", "value"] if "roberta" in model_name.lower() else ["c_attn"]

# =========================================================
# B: Comprehensive Injection (Attention + MLP Blocks)
#  =========================================================
# if "roberta" in model_name.lower():
#      target_mods = ["query", "key", "value", "dense"]
# else:
#      target_mods = ["c_attn", "c_proj", "c_fc"]

# ---------------------------------------------------------
# DO NOT MODIFY BELOW THIS LINE
# ---------------------------------------------------------
if 'target_mods' not in locals():
    raise ValueError("Please uncomment either Path A or Path B to configure target_mods!")

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=target_mods
)

model = get_peft_model(base_model, peft_config)
fine_tuning_strategy = "lora"
print(f"LoRA adapters successfully injected into targets: {target_mods}")

# **Decision 3.1 - Justification (Fill this out)**
**Which LORA adapters you choose (A or B):**

**Why you choosed this option:** [Provide your technical defense]

**Why you rejected the other option:** [Explain why the alternative failed or was suboptimal for this scenario]

In [ ]:
## Cell 6

# ---------------------------------------------------------
# Inspect the active parameter count resulting from your choice
# ---------------------------------------------------------
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
trainable_pct = 100 * trainable_params / all_params

print(f"Total Parameters     : {all_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")
print(f"Trainable Percentage : {trainable_pct:.4f}%")

# **Decision 3 - Justification (Fill this out)**
**Which Finetuning method you choose (A or B):**

**Why you choosed this option:** [Provide your technical defense]

**Why you rejected the other option:** [Explain why the alternative failed or was suboptimal for this scenario]

# **Decision 4: Classification Head Architecture**
The classification head takes the contextual token representation generated by the backbone Transformer and maps it to class probability logits. You must decide whether to use a minimal linear projection or a non-linear Multi-Layer Perceptron (MLP).

---

### A: Linear Classification Head
Directly projects the pooled sequence embedding to the target classes using a single matrix multiplication:
$$\text{Logits} = W x + b$$


### B: Multi-Layer Perceptron (MLP) Head
Passes the representation through an intermediate projection, non-linear activation (GELU), and dropout regularization before the final output layer:
$$\text{Logits} = W_2 \cdot \text{Dropout}(\text{GELU}(W_1 x + b_1)) + b_2$$


---

**Your Task:** Choose the classification head architecture that best fits your fine-tuning strategy and dataset scale. Uncomment your chosen path in the code cell below.

In [ ]:
import torch
import torch.nn as nn

# Determine hidden dimension from the loaded backbone configuration
hidden_dim = base_model.config.hidden_size if hasattr(base_model.config, "hidden_size") else base_model.config.n_embd
num_labels = 2
dropout_rate = 0.1

# =========================================================
# PATH A: Linear Classification Head (Minimalist & Regularized)
# =========================================================
# class CustomClassificationHead(nn.Module):
#     def __init__(self, hidden_dim, num_labels, dropout_rate):
#         super().__init__()
#         self.dropout = nn.Dropout(dropout_rate)
#         self.out_proj = nn.Linear(hidden_dim, num_labels)
#
#     def forward(self, features, **kwargs):
#         # Extract the <s> / [CLS] token representation at index 0 for encoders
#         x = features[:, 0, :] if features.dim() == 3 else features
#         x = self.dropout(x)
#         return self.out_proj(x)

# =========================================================
# PATH B: Non-Linear MLP Head (High Capacity)
# =========================================================
# class CustomClassificationHead(nn.Module):
#     def __init__(self, hidden_dim, num_labels, dropout_rate):
#         super().__init__()
#         intermediate_dim = hidden_dim // 2
#         self.dense = nn.Linear(hidden_dim, intermediate_dim)
#         self.act = nn.GELU()
#         self.dropout = nn.Dropout(dropout_rate)
#         self.out_proj = nn.Linear(intermediate_dim, num_labels)

#     def forward(self, features, **kwargs):
#         # Extract the <s> / [CLS] token representation at index 0 for encoders
#         x = features[:, 0, :] if features.dim() == 3 else features
#         x = self.dropout(self.act(self.dense(x)))
#         return self.out_proj(x)

# ---------------------------------------------------------
# DO NOT MODIFY BELOW THIS LINE
# ---------------------------------------------------------
if 'CustomClassificationHead' not in locals():
    raise ValueError("Please uncomment either Path A or Path B to define CustomClassificationHead!")

# Instantiate and attach the chosen head
custom_head = CustomClassificationHead(hidden_dim, num_labels, dropout_rate).to(device)

# Attach to appropriate model attribute depending on architecture
if hasattr(model, "classifier"):
    model.classifier = custom_head
elif hasattr(model, "score"):
    model.score = custom_head

print("Classification Head attached successfully.")

# **Decision 4 - Justification (Fill this out)**
**Which Classification head you choose (A or B):**

**Why you choosed this option:** [Provide your technical defense]

**Why you rejected the other option:** [Explain why the alternative failed or was suboptimal for this scenario]

# **Decision 5: Batch Size & Memory Optimization**
To effectively train the model and get smooth gradient updates, we need an effective batch size of 32.

You must achieve an effective batch size of 32 using an optimization strategy.

**A: Standard Batching**
Set the physical batch size to 32 and perform a forward pass, backward pass, and optimizer step for all 32 instances at once.
*Effective Batch = Batch Size (32) × Grad Accumulation Steps (1) = 32*

**B: Gradient Accumulation**
Set the physical batch size to a micro-batch of 4. Perform the forward and backward passes, but *accumulate* the gradients in memory. Only perform the optimizer step after 8 micro-batches have been processed.
*Effective Batch = Batch Size (4) × Grad Accumulation Steps (8) = 32*

**Your Task:** Select the configuration that prevents the GPU from crashing while maintaining the exact mathematical equivalent of a 32-batch update.

In [ ]:
from transformers import TrainingArguments

# =========================================================
# A: Standard Batching
# =========================================================
# per_device_batch_size = 32
# gradient_accumulation = 1

# =========================================================
# B: Gradient Accumulation
# =========================================================
# per_device_batch_size = 4
# gradient_accumulation = 8

# ---------------------------------------------------------
# DO NOT MODIFY BELOW THIS LINE
# ---------------------------------------------------------
if 'per_device_batch_size' not in locals():
    raise ValueError("Please uncomment either Path A or Path B to configure your batch settings!")

# Set learning rate dynamically based on earlier fine-tuning decision
# LoRA requires a higher learning rate (e.g., 2e-4) compared to standard fine-tuning (e.g., 2e-5)
current_lr = 2e-4 if fine_tuning_strategy == "lora" else 2e-5

training_args = TrainingArguments(
    output_dir="./medical_triage_model",
    per_device_train_batch_size=per_device_batch_size,
    gradient_accumulation_steps=gradient_accumulation,
    learning_rate=current_lr,
    num_train_epochs=3,
    logging_steps=10,
    fp16=True, # Use mixed precision to save VRAM and speed up training
    optim="adamw_torch",
    report_to="none" # Disable wandb/logging for this assignment
)

print("Training arguments configured successfully.")
print(f"Physical Batch Size : {per_device_batch_size}")
print(f"Effective Batch Size: {per_device_batch_size * gradient_accumulation}")

# **Dataset Preparation:**


In [ ]:
import pandas as pd
from datasets import Dataset

print("Downloading dataset directly from public storage...")

# Use a clean, public CSV dataset URL (e.g., a standard text classification dataset)
csv_url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv"

# Since it's a tab-separated file with 'label' and 'message' columns
df = pd.read_csv(csv_url, sep='\t', header=None, names=['label_str', 'text'])

# Map text labels ('ham'/'spam') to integer labels (0 and 1)
df['label'] = df['label_str'].map({'ham': 0, 'spam': 1})

# Keep only the columns we need and take a clean subset of 10,000 samples (or as many as available)
df = df[['text', 'label']].dropna()

# Let's slice to match your target size
train_df = df.iloc[:4500].reset_index(drop=True)
eval_df = df.iloc[4500:5000].reset_index(drop=True)

# Convert Pandas DataFrames into Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

num_labels = 2  # Binary classification (Ham vs Spam)

print(f"Dataset loaded successfully via direct CSV!")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

print("Tokenizing dataset...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# **Train the Model:**
You have successfully configured the architecture, fine-tuning strategy, classification head, and memory optimizations.





**Choose Appropriate number of taining Epochs:** [Any value between 1 to 10]

In [ ]:
number_of_epochs = 1

In [ ]:
from transformers import Trainer, TrainingArguments
import math

# ---------------------------------------------------------
# HYPERPARAMETERS
# Feel free to adjust the epochs and logging frequency
# ---------------------------------------------------------
num_epochs = number_of_epochs
logging_freq = 10

# Dynamically set learning rate based on fine-tuning strategy
current_lr = 2e-4 if fine_tuning_strategy == "lora" else 2e-5

training_args = TrainingArguments(
    output_dir="./medical_triage_model",
    per_device_train_batch_size=per_device_batch_size,   # Inherited from Decision 4
    gradient_accumulation_steps=gradient_accumulation,   # Inherited from Decision 4
    learning_rate=current_lr,
    num_train_epochs=num_epochs,

    # Logging & Evaluation Configuration
    logging_strategy="steps",
    logging_steps=logging_freq,
    eval_strategy="epoch",         # Evaluate at the end of each epoch

    # Checkpointing Configuration
    save_strategy="epoch",         # Save a checkpoint at the end of each epoch
    save_total_limit=1,            # Keep only the 1 most recent checkpoint to save disk space
    load_best_model_at_end=True,   # Automatically reload the best weights when training finishes

    fp16=True,                     # Use mixed precision to save VRAM
    optim="adamw_torch",
    report_to="none"               # Disable external telemetry (like WandB) for this assignment
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("Starting training pipeline... Watch the loss metrics below!")
try:
    trainer.train()
    print("\n✅ Training completed successfully! The best checkpoint has been loaded into memory.")
except RuntimeError as e:
    if "OutOfMemoryError" in str(e) or "CUDA out of memory" in str(e):
        print("\n❌ CUDA OUT OF MEMORY ERROR ❌")
        print("Your GPU ran out of VRAM. Restart your runtime, select the lower-memory paths, and try again.")
    else:
        raise e

# **Export and Download Checkpoint:**
Once training is complete, your tuned model weights are saved in the `./medical_triage_model` directory.


In [ ]:
import os
import shutil
from google.colab import files

output_dir = "./medical_triage_model"
zip_path = "./medical_triage_model_checkpoint.zip"

if os.path.exists(output_dir):
    print("Zipping the model checkpoint directory. This may take a moment depending on your fine-tuning strategy...")

    # Compress the directory into a zip file
    shutil.make_archive(zip_path.replace('.zip', ''), 'zip', output_dir)

    # Check the size of the generated zip file
    file_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"Zip archive created successfully! File size: {file_size_mb:.2f} MB")

    print("Triggering browser download...")
    files.download(zip_path)
else:
    print("❌ No checkpoint directory found. Did the training loop finish successfully?")

# **Do you think the above number of training Epochs is sufficient? Justify your answer(Use appropriate plots).**

# **Your answer here:**

**If you justified that the number of training epochs is insuffice, load the previously trained checkpoints and train again to more number of epochs as you wish.**

# **Choose "new number of Epochs":**

In [ ]:
new_num_epochs = 2

In [ ]:
import os
import zipfile
from transformers import Trainer, TrainingArguments

# =========================================================
# 1. UNZIP PREVIOUS CHECKPOINT (If uploaded back to Colab)
# =========================================================
zip_filename = "medical_triage_model_checkpoint.zip"
output_dir = "./medical_triage_model"

if os.path.exists(zip_filename) and not os.path.exists(output_dir):
    print(f"📦 Extracting {zip_filename}...")
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(output_dir)
    print("Extraction complete!")

# =========================================================
# 2. CONFIGURE NEW HYPERPARAMETERS (e.g., More Epochs)
# =========================================================
# Increase epochs to continue training further (e.g., from 3 to 5 total epochs)
new_num_epochs = new_num_epochs
logging_freq = 10

# Ensure learning rate variable is defined
current_lr = 2e-4 if 'fine_tuning_strategy' in locals() and fine_tuning_strategy == "lora" else 2e-5

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_batch_size,   # Inherited from Decision 5
    gradient_accumulation_steps=gradient_accumulation,   # Inherited from Decision 5
    learning_rate=current_lr,
    num_train_epochs=new_num_epochs,  # Updated epoch count

    # Logging & Evaluation Configuration
    logging_strategy="steps",
    logging_steps=logging_freq,
    eval_strategy="epoch",

    # Checkpointing Configuration
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,

    fp16=True,
    optim="adamw_torch",
    report_to="none"
)

# Re-initialize the Trainer with the current model state
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# =========================================================
# 3. FIND LATEST CHECKPOINT & RESUME TRAINING
# =========================================================
checkpoint_dir = None
if os.path.exists(output_dir):
    subdirs = [os.path.join(output_dir, d) for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if subdirs:
        checkpoint_dir = max(subdirs, key=os.path.getmtime)
        print(f"🔄 Resuming training from checkpoint: {checkpoint_dir}")
    else:
        print("📁 No checkpoint subfolders found. Starting training from scratch.")

print(f"🚀 Continuing training up to {new_num_epochs} total epochs...")
try:
    # resume_from_checkpoint loads the exact optimizer states, step numbers, and weights
    trainer.train(resume_from_checkpoint=checkpoint_dir)
    print("\n✅ Extended training completed successfully!")
except RuntimeError as e:
    if "OutOfMemoryError" in str(e) or "CUDA out of memory" in str(e):
        print("\n❌ CUDA OUT OF MEMORY ERROR ❌")
        print("Your GPU ran out of VRAM during continued training.")
    else:
        raise e

# **Model Evaluation & Performance Metrics:**
Now that training is complete (or loaded from a checkpoint), we must evaluate the model's generalization capabilities on the unseen validation set (500 samples).

Run the cell below to compute evaluation loss, accuracy, and F1-score.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("Running evaluation on the validation dataset...")

# Predict logits on the evaluation dataset
eval_results = trainer.predict(eval_dataset)
logits = eval_results.predictions
predictions = np.argmax(logits, axis=-1)
true_labels = eval_results.label_ids

# Compute standard classification metrics
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='binary')

print("========================================")
print(f"📊 Validation Results:")
print(f"   - Eval Loss  : {eval_results.metrics.get('test_loss', 0.0):.4f}")
print(f"   - Accuracy   : {accuracy * 100:.2f}%")
print(f"   - Precision  : {precision:.4f}")
print(f"   - Recall     : {recall:.4f}")
print(f"   - F1-Score   : {f1:.4f}")
print("========================================")